# Improved Logistic Regression & Multi-Model TF-IDF Ranker - Smart MCQ Solver

### Overview & Improvements
This notebook improves upon the baseline **Logistic Regression TF-IDF** solution for the **Smart MCQ Solver Challenge**.

- **Key Upgrades**:
  1. **Dual Token Vectorization**: Combines Word-level TF-IDF (`ngram_range=(1,3)`) + Character-level TF-IDF (`ngram_range=(2,5)`) to capture both semantic n-grams and morphological subwords.
  2. **Lexical & Contextual Feature Engineering**: Added word overlap, Jaccard similarity, and relative length ratio between question prompt and option text.
  3. **Multi-Model Ensemble**: Combines **Logistic Regression** (L2 penalized), **Ridge Classifier**, and **SGD Classifier** (probabilistic) across 5 Group K-Folds.
  4. **Robust Path & Wandb Integration**: Handles local and Kaggle environments seamlessly without crashing if Wandb is offline.
  5. **MAP@3 Metric & Submission Verification**: Computes Out-Of-Fold MAP@3 score and formats submission automatically.

---


In [ ]:
# Cell 1: Core Imports & Setup
import os
import re
import warnings
import numpy as np
import pandas as pd
from scipy.sparse import hstack, csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression, RidgeClassifier, SGDClassifier
from sklearn.model_selection import GroupKFold

warnings.filterwarnings('ignore')
print("Core libraries imported successfully!")


In [ ]:
# Cell 2: Robust Weights & Biases (Wandb) Setup
try:
    import wandb
    WANDB_API_KEY = "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"
    WANDB_ENTITY = "models-indian-institute-of-technology-madras481" 
    WANDB_PROJECT = "23f3001092-t22026"

    wandb.login(key=WANDB_API_KEY)
    USE_WANDB = True
    print("Wandb initialized successfully!")
except Exception as e:
    USE_WANDB = False
    print(f"Wandb disabled or running offline ({e})")


In [ ]:
# Cell 3: Flexible Data Loading
def load_datasets():
    train_candidates = [
        "train.csv",
        "/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv",
        "/kaggle/input/smart-mcq-solver-challenge/train.csv"
    ]
    test_candidates = [
        "test.csv",
        "/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv",
        "/kaggle/input/smart-mcq-solver-challenge/test.csv"
    ]
    
    train_path = next((p for p in train_candidates if os.path.exists(p)), train_candidates[0])
    test_path = next((p for p in test_candidates if os.path.exists(p)), test_candidates[0])
    
    df_train = pd.read_csv(train_path).fillna('')
    df_test = pd.read_csv(test_path).fillna('')
    
    return df_train, df_test

train, test = load_datasets()
OPTION_COLS = ['A', 'B', 'C', 'D', 'E']

print(f"Train shape: {train.shape}")
print(f"Test shape : {test.shape}")
train.head(3)


In [ ]:
# Cell 4: MAP@3 Metric Definition & Ranker Helper
def apk(actual, predicted, k=3):
    if actual in predicted[:k]:
        return 1.0 / (predicted[:k].index(actual) + 1.0)
    return 0.0

def mapk(actuals, preds, k=3):
    return float(np.mean([apk(a, p, k) for a, p in zip(actuals, preds)]))

def rank_preds(pw, scores, orig_df):
    pw_temp = pw.copy()
    pw_temp['score'] = scores
    
    ranked = pw_temp.sort_values(['qid', 'score'], ascending=[True, False])
    grouped = ranked.groupby('qid')['option'].apply(lambda opts: list(opts)[:3]).to_dict()
    
    return [grouped.get(qid, ['A', 'B', 'C']) for qid in orig_df['id']]


In [ ]:
# Cell 5: Advanced Pairwise Data Preparation & Lexical Features
def make_pairwise_dataset(df, is_train=True):
    rows = []
    for _, r in df.iterrows():
        qid = r['id']
        prompt = str(r['prompt'])
        p_words = set(re.findall(r'\w+', prompt.lower()))
        ans = r.get('answer', None)
        
        for c in OPTION_COLS:
            opt_str = str(r[c])
            o_words = set(re.findall(r'\w+', opt_str.lower()))
            
            overlap = len(p_words.intersection(o_words))
            jaccard = overlap / max(len(p_words.union(o_words)), 1)
            len_ratio = len(opt_str) / max(len(prompt), 1)
            
            combined_text = f"{prompt} [SEP] {opt_str}"
            
            row = {
                'qid': qid,
                'option': c,
                'text': combined_text,
                'overlap': overlap,
                'jaccard': jaccard,
                'len_ratio': len_ratio
            }
            if is_train:
                row['label'] = 1 if (ans == c) else 0
            rows.append(row)
            
    return pd.DataFrame(rows)

pw_train = make_pairwise_dataset(train, is_train=True)
pw_test = make_pairwise_dataset(test, is_train=False)

print(f"Pairwise Train Shape: {pw_train.shape}")
print(f"Pairwise Test Shape : {pw_test.shape}")
pw_train.head(5)


In [ ]:
# Cell 6: Word & Character TF-IDF Vectorization
print("Fitting Word and Character TF-IDF Vectorizers...")

tfidf_word = TfidfVectorizer(max_features=80000, ngram_range=(1, 3), sublinear_tf=True, analyzer='word')
tfidf_char = TfidfVectorizer(max_features=50000, ngram_range=(2, 5), sublinear_tf=True, analyzer='char_wb')

combined_corpus = pd.concat([pw_train['text'], pw_test['text']])
tfidf_word.fit(combined_corpus)
tfidf_char.fit(combined_corpus)

Xw_tr = tfidf_word.transform(pw_train['text'])
Xw_te = tfidf_word.transform(pw_test['text'])

Xc_tr = tfidf_char.transform(pw_train['text'])
Xc_te = tfidf_char.transform(pw_test['text'])

X_num_tr = csr_matrix(pw_train[['overlap', 'jaccard', 'len_ratio']].values)
X_num_te = csr_matrix(pw_test[['overlap', 'jaccard', 'len_ratio']].values)

X_tr = hstack([Xw_tr, Xc_tr, X_num_tr])
X_te = hstack([Xw_te, Xc_te, X_num_te])

y_tr = pw_train['label'].values
groups = pw_train['qid'].values

print(f"Final Train Sparse Feature Matrix: {X_tr.shape}")
print(f"Final Test Sparse Feature Matrix : {X_te.shape}")


In [ ]:
# Cell 7: 5-Fold Cross-Validation with Model Ensembling
gkf = GroupKFold(n_splits=5)

oof_lr = np.zeros(len(pw_train))
oof_ridge = np.zeros(len(pw_train))
oof_sgd = np.zeros(len(pw_train))

tst_lr = np.zeros(len(pw_test))
tst_ridge = np.zeros(len(pw_test))
tst_sgd = np.zeros(len(pw_test))

if USE_WANDB:
    run = wandb.init(
        entity=WANDB_ENTITY,
        project=WANDB_PROJECT,
        name="improved-logistic-ensemble-tfidf",
        config={
            "models": ["LogisticRegression", "RidgeClassifier", "SGDClassifier"],
            "C_lr": 6.0,
            "alpha_ridge": 1.0,
            "max_features_word": 80000,
            "max_features_char": 50000,
            "n_splits": 5,
        }
    )

print("Starting 5-Fold Training Loop...")

for fold, (ti, vi) in enumerate(gkf.split(X_tr, y_tr, groups)):
    print(f" Fold {fold+1}/5 training...")
    
    # Model 1: Logistic Regression
    lr = LogisticRegression(C=6.0, max_iter=2000)
    lr.fit(X_tr[ti], y_tr[ti])
    oof_lr[vi] = lr.predict_proba(X_tr[vi])[:, 1]
    tst_lr += lr.predict_proba(X_te)[:, 1] / 5.0
    
    # Model 2: Ridge Classifier
    ridge = RidgeClassifier(alpha=1.0)
    ridge.fit(X_tr[ti], y_tr[ti])
    oof_ridge[vi] = ridge.decision_function(X_tr[vi])
    tst_ridge += ridge.decision_function(X_te) / 5.0
    
    # Model 3: SGD Classifier
    sgd = SGDClassifier(loss='log_loss', max_iter=1000, random_state=42)
    sgd.fit(X_tr[ti], y_tr[ti])
    oof_sgd[vi] = sgd.predict_proba(X_tr[vi])[:, 1]
    tst_sgd += sgd.predict_proba(X_te)[:, 1] / 5.0

# Blended Ensembling (50% LR + 30% Ridge + 20% SGD)
oof_blend = 0.50 * oof_lr + 0.30 * oof_ridge + 0.20 * oof_sgd
tst_blend = 0.50 * tst_lr + 0.30 * tst_ridge + 0.20 * tst_sgd

map_lr = mapk(train['answer'].tolist(), rank_preds(pw_train, oof_lr, train))
map_ridge = mapk(train['answer'].tolist(), rank_preds(pw_train, oof_ridge, train))
map_sgd = mapk(train['answer'].tolist(), rank_preds(pw_train, oof_sgd, train))
oof_map3 = mapk(train['answer'].tolist(), rank_preds(pw_train, oof_blend, train))

print("\n==================================================")
print("             MODEL VALIDATION SCORES")
print("==================================================")
print(f" 1. Logistic Regression MAP@3 : {map_lr:.4f}")
print(f" 2. Ridge Classifier MAP@3    : {map_ridge:.4f}")
print(f" 3. SGD Classifier MAP@3      : {map_sgd:.4f}")
print(f" --------------------------------------------------")
print(f" BEST ENSEMBLE BLEND MAP@3 SCORE: {oof_map3:.4f}")
print("==================================================\n")

if USE_WANDB:
    wandb.log({"oof_map3": oof_map3, "map_lr": map_lr, "map_ridge": map_ridge, "map_sgd": map_sgd})
    wandb.finish()


In [ ]:
# Cell 8: Final Submission Generation & Verification
test_preds = rank_preds(pw_test, tst_blend, test)

submission = pd.DataFrame({
    'ID': test['id'],
    'Prediction': [' '.join(p) for p in test_preds]
})

submission.to_csv('submission.csv', index=False)
print("Successfully generated submission.csv!")

assert len(submission) == len(test), f"Expected {len(test)} rows, got {len(submission)}"
assert submission['Prediction'].str.match(r'^[A-E] [A-E] [A-E]$').all(), "Invalid prediction format!"

print("\nSubmission Preview:")
submission.head(10)
